# k-Nearest Neighbors — Exercises

Companion to the note [k-Nearest Neighbors](k-Nearest%20Neighbors.md).

Practise majority-vote and mean prediction, distance metrics, feature scaling, distance weighting, choosing $k$ by cross-validation, the curse of dimensionality and the Cover–Hart bound, by hand and from scratch in NumPy.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.spatial.distance import cdist
from scipy.stats import norm
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
rng = np.random.default_rng(0)
Xir, yir = load_iris(return_X_y=True)
perm = rng.permutation(150)
Xtr, ytr, Xte, yte = Xir[perm[:100]], yir[perm[:100]], Xir[perm[100:]], yir[perm[100:]]

## Part A · Concepts

### Exercise 1 · A lazy learner
*🧠 Concept · ★☆☆*

The note calls k-NN a *lazy learner*. With $N$ training points in $d$ dimensions and brute-force search, what is the cost of
(a) training, (b) one prediction, (c) storing the model? Contrast with [[Logistic Regression]]. When is lazy a good trade-off?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What does 'fit' actually do for k-NN?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Training: $O(1)$ beyond storing the data (or building a k-d/ball tree, $O(N\log N)$). (b) Prediction: $O(Nd)$ to compute all distances plus selecting the $k$ smallest
($O(N)$ with `argpartition`). (c) Memory: the whole training set, $O(Nd)$.
Logistic regression is the opposite: expensive fit, $O(d)$ prediction and storage. Lazy learning pays off when data change often (just append),
when there are few queries, or when the decision boundary is too irregular for a parametric model.

</details>

### Exercise 2 · The role of $k$
*🧠 Concept · ★☆☆*

(a) What is the training error of 1-NN (no duplicate points)? Why is it a useless estimate of test error?
(b) What does k-NN predict when $k=N$?
(c) Relate small/large $k$ to variance/bias ([[Bias-Variance Tradeoff]]). Why is an odd $k$ preferred for binary classification?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Each training point is its own nearest neighbour.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $0$: every training point is its own neighbour, so it is perfectly memorised; test error can be much higher (overfitting).
(b) The majority class (or the global mean in regression) for every query: a constant model.
(c) Small $k$: very flexible, jagged boundary that changes a lot with the sample → high variance, low bias. Large $k$: smooth boundary, averages over
far-away points → high bias, low variance. Odd $k$ avoids ties in two-class votes. Choose $k$ by [[Cross-Validation and Model Selection]].

</details>

### Exercise 3 · Distances concentrate
*🧠 Concept · ★★☆*

Explain the *curse of dimensionality* for k-NN in two ways: (a) neighbourhoods are no longer local, (b) the ratio
$\frac{d_{\max}-d_{\min}}{d_{\min}}$ shrinks. Why does real data often suffer less than uniform random data?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

For i.i.d. coordinates, $\lVert x-x'\rVert^2$ is a sum of $d$ independent terms: its mean grows like $d$, its standard deviation like $\sqrt d$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) To capture a fixed fraction of uniformly spread data the neighbourhood must cover most of the range of every coordinate (Exercise 9), so 'nearest'
neighbours are far away and not similar. (b) The squared distance has mean $\propto d$ but spread $\propto\sqrt d$, so the relative contrast
$\to0$: all points look equally far, and the ranking of neighbours becomes noise-dominated.
Real data usually lies near a low-dimensional manifold (intrinsic dimension $\ll d$), so distances stay informative; dimensionality reduction
([[PCA]]) or metric learning help further.

</details>

### Exercise 4 · Making k-NN fast
*🧠 Concept · ★★☆*

The note lists k-d trees, ball trees, LSH and approximate NN (FAISS, HNSW). Explain in one or two sentences each how a k-d tree prunes the search,
why it degrades in high dimension, and what approximate methods give up.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A k-d tree splits on one coordinate at a time. When can a whole subtree be skipped?

</details>

<details>
<summary><b>✅ Solution</b></summary>

**k-d tree**: recursively split on the median of one coordinate. During search, a subtree is skipped if the distance from the query to its splitting
hyperplane (bounding box) exceeds the current $k$-th best distance. In high $d$ almost every box is within that distance (concentration again), so nearly all
leaves are visited and it is no better than brute force (rule of thumb: useful for $d\lesssim 20$). **Ball trees** use hyperspheres and cope a bit better with structured data.
**LSH / HNSW / FAISS** return neighbours that are *probably* among the nearest (recall < 100%) in exchange for sub-linear query time on millions of vectors.

</details>

## Part B · By hand

### Exercise 5 · Majority vote in 1-D
*✍️ By hand · ★☆☆*

Training points on a line with labels A=0, B=1:

| $x$ | 1 | 2 | 4 | 5 | 7 | 8 |
|---|---|---|---|---|---|---|
| label | A | A | B | A | B | B |

Predict the class of $x_q=4.6$ with $k=1$, $k=3$ and $k=5$ (Euclidean distance).

In [ ]:
pred_k1 = None  # 0 for A, 1 for B
pred_k3 = None
pred_k5 = None

_x = np.array([1,2,4,5,7,8.])[:, None]; _y = np.array([0,0,1,0,1,1])
for _k, _g in [(1, pred_k1), (3, pred_k3), (5, pred_k5)]:
    check(f'k={_k}', _g, KNeighborsClassifier(_k).fit(_x, _y).predict([[4.6]])[0])

<details>
<summary><b>💡 Hint</b></summary>

Distances: $3.6, 2.6, 0.6, 0.4, 2.4, 3.4$. Sort them.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Sorted: $5$ (A, 0.4), $4$ (B, 0.6), $7$ (B, 2.4), $2$ (A, 2.6), $8$ (B, 3.4), $1$ (A, 3.6).
$k=1$: A. $k=3$: A, B, B → **B**. $k=5$: A, B, B, A, B → **B**. Predictions flip with $k$, which is exactly the variance of small $k$.

```python
pred_k1, pred_k3, pred_k5 = 0, 1, 1
```

</details>

### Exercise 6 · k-NN regression, plain and weighted
*✍️ By hand · ★☆☆*

Data $x=(1,2,4,5,7)$, $y=(2,3,6,5,9)$. Predict at $x_q=3.5$ with $k=3$ using (a) the plain mean and (b) the note's weighted vote $w_i=1/d_i$
(weighted mean $\sum w_iy_i/\sum w_i$).

In [ ]:
yhat_mean = None
yhat_weighted = None

_x = np.array([1,2,4,5,7.])[:, None]; _y = np.array([2,3,6,5,9.])
check('plain mean', yhat_mean, KNeighborsRegressor(3).fit(_x, _y).predict([[3.5]])[0], tol=1e-3)
check('distance weighted', yhat_weighted, KNeighborsRegressor(3, weights='distance').fit(_x, _y).predict([[3.5]])[0], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

The three neighbours are $x=4$ ($d=0.5$), $x=2$ and $x=5$ (both $d=1.5$).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $(6+3+5)/3\approx4.667$. (b) Weights $2,\ 2/3,\ 2/3$: $\frac{2\cdot6+\tfrac23\cdot3+\tfrac23\cdot5}{2+\tfrac43}=\frac{17.33}{3.33}=5.2$.
Weighting pulls the prediction towards the closest point. Note $w_i=1/d_i$ is infinite at $d_i=0$; implementations return that point's label.

```python
yhat_mean = 14 / 3
yhat_weighted = 5.2
```

</details>

### Exercise 7 · Four distances
*✍️ By hand · ★☆☆*

For $a=(1,2,3)$ and $b=(2,0,4)$ compute the Euclidean, Manhattan and cosine distance ($1-\cos\theta$), and the Mahalanobis distance
$\sqrt{(a-b)^\top\Sigma^{-1}(a-b)}$ with $\Sigma=\operatorname{diag}(1,4,1)$.

In [ ]:
d_euc = None
d_man = None
d_cos = None
d_mah = None

_a, _b = np.array([[1,2,3.]]), np.array([[2,0,4.]])
check('Euclidean', d_euc, cdist(_a, _b)[0, 0], tol=1e-3)
check('Manhattan', d_man, cdist(_a, _b, 'cityblock')[0, 0])
check('cosine', d_cos, cdist(_a, _b, 'cosine')[0, 0], tol=1e-3)
check('Mahalanobis', d_mah, cdist(_a, _b, 'mahalanobis', VI=np.linalg.inv(np.diag([1,4,1.])))[0, 0], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$a-b=(-1,2,-1)$; $a^\top b=14$, $\lVert a\rVert=\sqrt{14}$, $\lVert b\rVert=\sqrt{20}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Euclidean $\sqrt{1+4+1}=\sqrt6\approx2.449$. Manhattan $1+2+1=4$. Cosine $1-\frac{14}{\sqrt{280}}\approx0.1633$.
Mahalanobis $\sqrt{1+4/4+1}=\sqrt3\approx1.732$: the second coordinate has variance 4, so its difference counts less.
Mahalanobis with the data covariance is Euclidean distance after *whitening* the features.

```python
d_euc = np.sqrt(6)
d_man = 4
d_cos = 1 - 14 / np.sqrt(280)
d_mah = np.sqrt(3)
```

</details>

### Exercise 8 · Scaling changes the neighbours
*✍️ By hand · ★☆☆*

Features: age (years) and income (€). Query $q=(30, 50\,000)$, candidates $P_1=(31, 52\,000)$ and $P_2=(60, 50\,500)$.
(a) Which is nearer in raw Euclidean distance? (b) After dividing by standard deviations $\sigma_{\text{age}}=10$, $\sigma_{\text{inc}}=10\,000$?
Store the index (1 or 2) of the nearest point in each case, and the scaled distance from $q$ to $P_1$.

In [ ]:
nearest_raw = None
nearest_scaled = None
d_scaled_P1 = None

_q, _P = np.array([30, 50000.]), np.array([[31, 52000], [60, 50500.]])
check('raw', nearest_raw, np.argmin(np.linalg.norm(_P - _q, axis=1)) + 1)
check('scaled', nearest_scaled, np.argmin(np.linalg.norm((_P - _q) / [10, 10000], axis=1)) + 1)
check('scaled distance to P1', d_scaled_P1, np.linalg.norm((_P[0] - _q) / [10, 10000]), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Raw: income differences (thousands) dwarf age differences (tens).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Raw: $d(q,P_1)=\sqrt{1+2000^2}\approx2000$, $d(q,P_2)=\sqrt{30^2+500^2}\approx500.9$, so $P_2$, a 60-year-old, is 'nearest' purely because of the € units.
(b) Scaled: $P_1\to(0.1,0.2)$, distance $\sqrt{0.05}\approx0.224$; $P_2\to(3,0.05)$, distance $\approx3.0$. Now $P_1$ is nearest.
Always standardise before k-NN ([[Data Preprocessing]]).

```python
nearest_raw = 2
nearest_scaled = 1
d_scaled_P1 = np.sqrt(0.05)
```

</details>

### Exercise 9 · How big is a 'local' neighbourhood?
*✍️ By hand · ★★☆*

Data are uniform in the unit hypercube $[0,1]^d$. A sub-cube that captures a fraction $r$ of the data has edge length $e_d(r)=r^{1/d}$.
Compute $e_{10}(0.1)$ and $e_{100}(0.1)$. What does this mean for 'nearest' neighbours?

In [ ]:
e10 = None
e100 = None

check('e_10(0.1)', e10, 0.1**(1/10), tol=1e-3)
check('e_100(0.1)', e100, 0.1**(1/100), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Volume of a cube with edge $e$ is $e^d$; set it to $r$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$e^d=r\Rightarrow e=r^{1/d}$. $e_{10}(0.1)=0.1^{0.1}\approx0.794$, $e_{100}(0.1)\approx0.977$.
To average over just 10% of the data in 10-D you must span almost 80% of each coordinate's range: the neighbourhood is not local any more.

```python
e10 = 0.1 ** 0.1
e100 = 0.1 ** 0.01
```

</details>

### Exercise 10 · The Cover–Hart bound
*✍️ By hand · ★★☆*

For two classes, Cover & Hart showed that as $N\to\infty$ the 1-NN error $E_{1}$ satisfies $E^*\le E_{1}\le 2E^*(1-E^*)$, where $E^*$ is the Bayes error.
(a) Evaluate the upper bound for $E^*=0.1$. (b) Give the intuition for the factor ~2 with a single point $x$ where $p(1\mid x)=\eta$.

In [ ]:
bound_1nn = None

check('upper bound', bound_1nn, 2*0.1*0.9)

<details>
<summary><b>💡 Hint</b></summary>

As $N\to\infty$ the nearest neighbour $x'$ sits at essentially the same location as $x$, with an independent label drawn from the same $\eta$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $2\cdot0.1\cdot0.9=0.18$, below the looser '$2E^*$' $=0.2$ of the note.
(b) At $x$, the query label and the neighbour's label are independent Bernoulli($\eta$). 1-NN errs when they differ: $P=2\eta(1-\eta)$.
The Bayes classifier errs with $\min(\eta,1-\eta)=:e$, and $2\eta(1-\eta)=2e(1-e)\le 2e$. Averaging over $x$ (Jensen) gives the bound. You will check it by simulation in Exercise 17.

```python
bound_1nn = 0.18
```

</details>

## Part C · Code from scratch

### Exercise 11 · Vectorised pairwise distances
*💻 Code · ★☆☆*

Implement `pairwise_sq_dists(A, B)` returning the $(n_A, n_B)$ matrix of squared Euclidean distances **without Python loops**, using
$\lVert a-b\rVert^2=\lVert a\rVert^2+\lVert b\rVert^2-2a^\top b$.

In [ ]:
def pairwise_sq_dists(A, B):
    # TODO
    return None

D_test = pairwise_sq_dists(Xte, Xtr)

check('matches scipy cdist', D_test, cdist(Xte, Xtr, 'sqeuclidean'), tol=1e-8)

<details>
<summary><b>💡 Hint</b></summary>

`(A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T`; clip tiny negatives caused by round-off with `np.maximum(., 0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The matrix product does the heavy lifting (BLAS), which is orders of magnitude faster than nested loops. Round-off can make entries slightly negative, hence the clip.

```python
def pairwise_sq_dists(A, B):
    D = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    return np.maximum(D, 0)

D_test = pairwise_sq_dists(Xte, Xtr)
```

</details>

### Exercise 12 · k-NN classifier
*💻 Code · ★☆☆*

Implement `knn_predict(Xtr, ytr, Xq, k)` with majority vote (ties → smallest label). Use it on the Iris split from the setup with $k=5$ and compare
with `KNeighborsClassifier`.

In [ ]:
def knn_predict(Xtr, ytr, Xq, k):
    # TODO
    return None

pred_knn = knn_predict(Xtr, ytr, Xte, 5)

check('agreement with sklearn', None if pred_knn is None else np.mean(pred_knn == KNeighborsClassifier(5).fit(Xtr, ytr).predict(Xte)), 1.0)

<details>
<summary><b>💡 Hint</b></summary>

`idx = np.argsort(D, axis=1)[:, :k]`, then `np.bincount(ytr[row]).argmax()` per row (argmax returns the first maximum, i.e. the smallest label).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Test accuracy on this split is about 0.98. `np.argpartition` instead of a full sort gives $O(N)$ selection per query.

```python
def knn_predict(Xtr, ytr, Xq, k):
    D = pairwise_sq_dists(Xq, Xtr)
    idx = np.argsort(D, axis=1, kind='stable')[:, :k]
    return np.array([np.bincount(ytr[row]).argmax() for row in idx])

pred_knn = knn_predict(Xtr, ytr, Xte, 5)
print('test accuracy', np.mean(pred_knn == yte))
```

</details>

### Exercise 13 · Distance-weighted k-NN regression
*💻 Code · ★★☆*

Implement `knn_regress(Xtr, ytr, Xq, k, weighted)` (plain mean or $1/d$ weights; if a distance is exactly 0, return that training target).
Test on noisy $y=\sin x$ data and compare with `KNeighborsRegressor(weights='distance')`.

In [ ]:
def knn_regress(Xtr, ytr, Xq, k, weighted=False):
    # TODO
    return None

xs_tr = np.sort(rng.uniform(0, 6, 60))[:, None]
ys_tr = np.sin(xs_tr[:, 0]) + 0.2 * rng.normal(size=60)
xs_q = np.linspace(0, 6, 25)[:, None]
yq_w = knn_regress(xs_tr, ys_tr, xs_q, 4, weighted=True)

check('matches sklearn (distance weights)', yq_w, KNeighborsRegressor(4, weights='distance').fit(xs_tr, ys_tr).predict(xs_q))

<details>
<summary><b>💡 Hint</b></summary>

With `d = np.sqrt(D[i, idx])`: `w = 1/d`; `np.sum(w * y) / np.sum(w)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Weighting by inverse distance reduces the 'flat steps' of plain k-NN regression and makes predictions interpolate the training targets exactly at training points.

```python
def knn_regress(Xtr, ytr, Xq, k, weighted=False):
    D = np.sqrt(pairwise_sq_dists(Xq, Xtr))
    out = np.empty(len(Xq))
    for i in range(len(Xq)):
        idx = np.argsort(D[i], kind='stable')[:k]
        d = D[i, idx]
        if not weighted:
            out[i] = ytr[idx].mean()
        elif np.any(d == 0):
            out[i] = ytr[idx][d == 0].mean()
        else:
            w = 1 / d
            out[i] = np.sum(w * ytr[idx]) / np.sum(w)
    return out

xs_tr = np.sort(rng.uniform(0, 6, 60))[:, None]
ys_tr = np.sin(xs_tr[:, 0]) + 0.2 * rng.normal(size=60)
xs_q = np.linspace(0, 6, 25)[:, None]
yq_w = knn_regress(xs_tr, ys_tr, xs_q, 4, weighted=True)
```

</details>

### Exercise 14 · Choosing $k$ with cross-validation
*💻 Code · ★★☆*

On breast-cancer data, run 5-fold CV (`KFold(5)`, no shuffling) for $k\in\{1,3,5,\dots,21\}$ using your `knn_predict`, standardising **inside each fold**
(fit mean/std on the training folds only). Store the mean CV accuracies in `cv_acc` (array, same order as `ks`) and the best $k$ in `best_k`.

In [ ]:
Xbc, ybc = load_breast_cancer(return_X_y=True)
ks = list(range(1, 22, 2))
cv_acc = None
best_k = None

_ref = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)), Xbc, ybc, cv=KFold(5)).mean() for k in ks]
check('CV accuracies', cv_acc, _ref, tol=1e-9)
check('best k', best_k, ks[int(np.argmax(_ref))])

<details>
<summary><b>💡 Hint</b></summary>

For each split: `mu, sd = X[tr].mean(0), X[tr].std(0)`; transform both parts with these.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Here the curve is fairly flat (0.95–0.96), peaking at $k=9$ and slowly falling for larger $k$ as bias grows; small $k$ is noisier across folds. Scaling inside the folds avoids
leaking test-fold statistics into training (see [[Cross-Validation and Model Selection]]).

```python
Xbc, ybc = load_breast_cancer(return_X_y=True)
ks = list(range(1, 22, 2))
cv_acc = np.zeros(len(ks))
for tr, va in KFold(5).split(Xbc):
    mu, sd = Xbc[tr].mean(0), Xbc[tr].std(0)
    A, B = (Xbc[tr] - mu) / sd, (Xbc[va] - mu) / sd
    for j, k in enumerate(ks):
        cv_acc[j] += np.mean(knn_predict(A, ybc[tr], B, k) == ybc[va]) / 5
best_k = ks[int(np.argmax(cv_acc))]
print(np.round(cv_acc, 4), 'best k =', best_k)
```

</details>

### Exercise 15 · Scaling matters (wine data)
*💻 Code · ★☆☆*

`load_wine` has features on very different scales (proline is in the hundreds/thousands). Compute the 5-fold CV accuracy (sklearn is fine here)
of 5-NN without scaling (`acc_raw`) and with a `StandardScaler` in a pipeline (`acc_scaled`).

In [ ]:
Xw, yw = load_wine(return_X_y=True)
print(Xw.std(0).round(2))
acc_raw = None
acc_scaled = None

check('scaling helps by > 15 points', None if acc_raw is None else acc_scaled - acc_raw > 0.15, True)

<details>
<summary><b>💡 Hint</b></summary>

`cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(5)), Xw, yw, cv=5).mean()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Raw ≈ 0.69 vs scaled ≈ 0.95: without scaling the distance is essentially the proline difference alone and the other 12 features are ignored.

```python
Xw, yw = load_wine(return_X_y=True)
acc_raw = cross_val_score(KNeighborsClassifier(5), Xw, yw, cv=5).mean()
acc_scaled = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(5)), Xw, yw, cv=5).mean()
print(acc_raw, acc_scaled)
```

</details>

### Exercise 16 · Seeing the curse of dimensionality
*💻 Code · ★★☆*

For $d\in\{2,10,100,1000\}$ draw 500 uniform points in $[0,1]^d$ and one query point, and compute the relative contrast
$\frac{d_{\max}-d_{\min}}{d_{\min}}$ of the query's distances. Store the four values in `contrast` (list).

In [ ]:
dims = [2, 10, 100, 1000]
contrast = None

check('contrast shrinks with d', None if contrast is None else bool(np.all(np.diff(contrast) < 0)) and contrast[-1] < 0.3, True)

<details>
<summary><b>💡 Hint</b></summary>

Use a fresh `np.random.default_rng(1)`; `np.linalg.norm(P - q, axis=1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Typical values: ≈ 30 for $d=2$, ≈ 2.5 for $d=10$, ≈ 0.4 for $d=100$, ≈ 0.1 for $d=1000$. In 1000-D the nearest point is only ~10% closer than the farthest one,
so 'nearest' carries little information.

```python
dims = [2, 10, 100, 1000]
r = np.random.default_rng(1)
contrast = []
for d in dims:
    P, q = r.uniform(size=(500, d)), r.uniform(size=d)
    dist = np.linalg.norm(P - q, axis=1)
    contrast.append((dist.max() - dist.min()) / dist.min())
print(np.round(contrast, 3))
```

</details>

### Exercise 17 · Checking Cover–Hart by simulation
*💻 Code · ★★★*

Classes are equally likely with $x\mid y=0\sim\mathcal N(-1,1)$ and $x\mid y=1\sim\mathcal N(1,1)$. (a) Compute the Bayes error $E^*$ (threshold at 0).
(b) Estimate the 1-NN test error with 20 000 training and 20 000 test points. Store both, and check $E^*\le E_1\le2E^*(1-E^*)$.

In [ ]:
E_bayes = None
E_1nn = None

check('Bayes error', E_bayes, norm.cdf(-1), tol=1e-4)
check('E* <= E_1nn <= 2E*(1-E*)  (with MC slack)', None if E_1nn is None else norm.cdf(-1) - 0.01 <= E_1nn <= 2*norm.cdf(-1)*(1-norm.cdf(-1)) + 0.01, True)

<details>
<summary><b>💡 Hint 1</b></summary>

$E^*=P(x>0\mid y=0)=\Phi(-1)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

In 1-D the nearest neighbour can be found fast with `np.searchsorted` on the sorted training points (compare the left and right candidate) or with `KNeighborsClassifier(1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$E^*=\Phi(-1)\approx0.159$, the bound is $2E^*(1-E^*)\approx0.267$, and the simulated 1-NN error is about $0.22$: inside the bound and well above Bayes.
The bound is asymptotic; for finite $N$ in higher dimensions the error can be worse.

```python
r = np.random.default_rng(3)
def sample(n):
    y = r.integers(0, 2, n)
    return r.normal(2 * y - 1, 1.0), y
xtr1, ytr1 = sample(20000)
xte1, yte1 = sample(20000)
E_bayes = norm.cdf(-1)
order = np.argsort(xtr1); xs, ys = xtr1[order], ytr1[order]
pos = np.clip(np.searchsorted(xs, xte1), 1, len(xs) - 1)
left_closer = np.abs(xte1 - xs[pos - 1]) <= np.abs(xte1 - xs[pos])
nn_label = np.where(left_closer, ys[pos - 1], ys[pos])
E_1nn = np.mean(nn_label != yte1)
print(E_bayes, E_1nn, 2 * E_bayes * (1 - E_bayes))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [k-Nearest Neighbors](k-Nearest%20Neighbors.md).*